# Youtube Data API: Channel con Pais
https://developers.google.com/youtube/v3/docs/channels

## Contenido:

* [Librerias](*Librerias)
* [Search](*Search)


<a name="Librerias"></a>
## Librerias

In [ ]:
import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## Channels + Pais

### Ejecucion

#### Parametros constantes

In [ ]:
URIc='https://www.googleapis.com/youtube/v3/channels?key='

# ======parametros requeridos=====

part='snippet' #parametro obligatorio

#### Parametros variables

In [ ]:
#=====filtros (especifique exactamente uno de los siguientes parámetros)=====

id=ListaIDs[0] # lista separada por comas de los ID de canales de YouTube (Maximo 50 Ids)

# =====parametros opcionales=====

pageToken='' #pagina especifica que se debe devolver

#### Solicitud

In [ ]:
r = requests.get(URIc+key+'&part='+part+'&id='+id)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en Produccion

#### Importando Comentarios o Respuestas de Comentarios

In [ ]:
# ====================== OJO Cambiar para cada Serie ======================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/RespuestasComentarios/\
Narcos/2022-09-14RC1(2022-09-13)RC2(2022-09-14)/RESULTS-RESPUESTASCOMENTARIOS2022-09-14 16:24:36.794723.csv' 

# ====================== OJO Cambiar para cada Serie ======================

Comentarios = pd.read_csv(path)
Comentarios

In [ ]:
autor='idAutorCanalComentarioRespuesta' #'idAutorCanalComentarioPrincipal', 'idAutorCanalComentarioRespuesta' en caso de respuestas de comentarios

In [ ]:
dataCM=Comentarios.loc[:,[autor]]
dataCM

#### Eliminando Autores repetidos

In [ ]:
CMclear=dataCM.drop_duplicates(subset=[autor]).astype(str)
CMclear.info()

#### Formato Lista de IDs de Videos

In [ ]:
iz=0
de=50
IdsAutor=list(CMclear[autor])
ent=math.ceil(len(IdsAutor)/50)
ListaIDs=[]
for i in range(0,ent):
  stringIDs=','.join(IdsAutor[iz:de])
  iz=iz+50
  de=de+50
  ListaIDs.append(stringIDs)
print(len(ListaIDs))

#### Parametros constantes

In [ ]:
URIc='https://www.googleapis.com/youtube/v3/channels?key='

# ======parametros requeridos=====

part='snippet' #parametro obligatorio
fields='pageInfo(*),nextPageToken(*),prevPageToken(*)\
,items(id,snippet(country))'

#### Parametros variables

In [ ]:
#=====filtros (especifique exactamente uno de los siguientes parámetros)=====

id=ListaIDs #lista de ids de autores

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ===== OJO Cambiar para cada Serie =====

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/PaisesCanales/RespuestasComentarios/Narcos/'

# ===== OJO Cambiar para cada Serie =====

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'PAISCOMENTARIOS'+str(datetime.now())

#### Constantes

In [ ]:
solicitud=0
espera=0 #valor de retardo para las solicitudes
ithid=0 # i-esimo id
solicitudmax=3000 #numero de solicitudes maximas (el limite es 10000)

#### Ejecucion

In [ ]:
datac= pd.DataFrame() # Dataframe de Resultados Totales
df = pd.DataFrame(columns=['Solicitud','ListaIDs','FechaEjecucion'\
                           ,'Respuesta'])
for id in ListaIDs[ithid:]:
  solicitud=solicitud+1
  time.sleep(espera)
  r = requests.get(URIc+key+'&part='+part+'&fields='+fields+'&id='+id)  
  print('solicitud',solicitud,r.reason)
  d = dict(r.json())
  items = d.get('items')
  data=pd.json_normalize(items)
  datac=pd.concat([datac, data], ignore_index=True)  
  df = df.append({'Solicitud':solicitud,'ListaIDs':id,'FechaEjecucion':str(datetime.now())\
                  ,'Respuesta':r.reason}, ignore_index=True)

print("Paises Terminados")

In [ ]:
df

In [ ]:
datac

In [ ]:
datac.info()

In [ ]:
datac['snippet.country'].unique()

In [ ]:
mergedinner= pd.merge(left=Comentarios,right=datac, left_on=autor, right_on='id', how="left")
mergedinner.info()

In [ ]:
mergedinner=mergedinner.drop(['id'], axis=1)
mergedinner

In [ ]:
mergedinner.info()

#### Estructurar y dar formato a Dataframe

##### Respuestas Comentarios

In [ ]:
columnas={'idComentarioPrincipal':'idComentarioPrincipal'\
          ,'idComentarioRespuesta':'idComentarioRespuesta','TextoComentarioRespuesta':'TextoComentarioRespuesta','ComentarioRespuestaModificado':'ComentarioRespuestaModificado','emojisComentarioRespuesta':'emojisComentarioRespuesta'\
          ,'URLsComentarioRespuesta':'URLsComentarioRespuesta','hashtagsComentarioRespuesta':'hashtagsComentarioRespuesta','MencionesComentarioRespuesta':'MencionesComentarioRespuesta'\
          ,'LikesComentarioRespuesta':'LikesComentarioRespuesta'\
          ,'FechaPublicacionComentarioRespuesta':'FechaPublicacionComentarioRespuesta'\
          ,'FechaActualizacionComentarioRespuesta':'FechaActualizacionComentarioRespuesta','NombreAutorComentarioRespuesta':'NombreAutorComentarioRespuesta'\
          ,'URLAutorCanalComentarioRespuesta':'URLAutorCanalComentarioRespuesta','idAutorCanalComentarioRespuesta':'idAutorCanalComentarioRespuesta'\
          ,'CalificacionEspectadorComentarioRespuesta':'CalificacionEspectadorComentarioRespuesta','CalificarComentarioRespuesta':'CalificarComentarioRespuesta'\
          ,'SolicitudComentarioRespuesta':'SolicitudComentarioRespuesta','snippet.country':'PaisComentarioRespuesta'}

columnasorden=['idComentarioPrincipal','idComentarioRespuesta','TextoComentarioRespuesta','ComentarioRespuestaModificado','emojisComentarioRespuesta','URLsComentarioRespuesta','hashtagsComentarioRespuesta','MencionesComentarioRespuesta'\
               ,'LikesComentarioRespuesta','FechaPublicacionComentarioRespuesta','FechaActualizacionComentarioRespuesta'\
               ,'NombreAutorComentarioRespuesta','PaisComentarioRespuesta','URLAutorCanalComentarioRespuesta','idAutorCanalComentarioRespuesta','CalificacionEspectadorComentarioRespuesta','CalificarComentarioRespuesta','SolicitudComentarioRespuesta']

mergedinner.rename(columns=columnas,inplace=True)
DATA=mergedinner.reindex(columns=columnasorden)
DATA

##### Comentarios

In [ ]:
columnas={'idVideo':'idVideo','idComentarioPrincipal':'idComentarioPrincipal'\
          ,'TextoComentarioPrincipal':'TextoComentarioPrincipal','ComentarioPrincipalModificado':'ComentarioPrincipalModificado','emojisComentarioPrincipal':'emojisComentarioPrincipal'\
          ,'URLsComentarioPrincipal':'URLsComentarioPrincipal','hashtagsComentarioPrincipal':'hashtagsComentarioPrincipal','MencionesComentarioPrincipal':'MencionesComentarioPrincipal','LikesComentarioPrincipal':'LikesComentarioPrincipal'\
          ,'ReplicasTotalesComentarioPrincipal':'ReplicasTotalesComentarioPrincipal','FechaPublicacionComentarioPrincipal':'FechaPublicacionComentarioPrincipal'\
          ,'FechaActualizacionComentarioPrincipal':'FechaActualizacionComentarioPrincipal','PublicoComentarioPrincipal':'PublicoComentarioPrincipal'\
          ,'ReplicarComentarioPrincipal':'ReplicarComentarioPrincipal','NombreAutorComentarioPrincipal':'NombreAutorComentarioPrincipal'\
          ,'URLAutorCanalComentarioPrincipal':'URLAutorCanalComentarioPrincipal','idAutorCanalComentarioPrincipal':'idAutorCanalComentarioPrincipal'\
          ,'CalificacionEspectadorComentarioPrincipal':'CalificacionEspectadorComentarioPrincipal','CalificarComentarioPrincipal':'CalificarComentarioPrincipal'\
          ,'SolicitudComentarioPrincipal':'SolicitudComentarioPrincipal','snippet.country':'PaisComentarioPrincipal'}

columnasorden=['idVideo','idComentarioPrincipal','TextoComentarioPrincipal','ComentarioPrincipalModificado','emojisComentarioPrincipal','URLsComentarioPrincipal','hashtagsComentarioPrincipal','MencionesComentarioPrincipal'\
               ,'LikesComentarioPrincipal','ReplicasTotalesComentarioPrincipal','FechaPublicacionComentarioPrincipal','FechaActualizacionComentarioPrincipal'\
               ,'PublicoComentarioPrincipal','ReplicarComentarioPrincipal','NombreAutorComentarioPrincipal','PaisComentarioPrincipal','URLAutorCanalComentarioPrincipal','idAutorCanalComentarioPrincipal','CalificacionEspectadorComentarioPrincipal','CalificarComentarioPrincipal','SolicitudComentarioPrincipal']

mergedinner.rename(columns=columnas,inplace=True)
DATA=mergedinner.reindex(columns=columnasorden)
DATA

#### Exportar Dataframe a CSV

In [ ]:
df.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)